# 02. Feature Engineering
DAY1 전략서의 피처를 `src/features.py`로 계산하고, **B1 기준**으로 선택 근거를 다시 확인한다.
- 원본 캐시(`data/cache/*.pkl`)가 있으면 새로 계산, 없으면 `data/features.csv`를 읽음
- B2 · B3는 일반화 점검용으로만 본다 (선택 · 튜닝에 사용하지 않음)

In [1]:
import os, sys
sys.path.append(os.path.abspath('../src')); os.chdir('..')
import numpy as np, pandas as pd
if os.path.exists('data/cache/b1.pkl'):
    from features import build_features
    df = build_features('data/cache', 'data/features.csv')
else:
    df = pd.read_csv('data/features.csv')
df.groupby('batch').size()

batch
b1    36
b2    39
b3    40
dtype: int64

## B1 내 상관 (선택 기준) vs 배치별 상관 (점검용)

In [2]:
F = ['dq_var','dq_min','dq_mean','fade_10_100','QD_2','IR_2','Tavg_100','chargetime_5','avgC']
tab = pd.DataFrame({b: g[F].corrwith(g['log_life'], method='spearman') for b, g in df.groupby('batch')})
tab.columns = ['B1 (선택 기준)', 'B2 (점검)', 'B3 (점검)']
tab['전체'] = df[F].corrwith(df['log_life'], method='spearman')
tab.round(2).sort_values('B1 (선택 기준)')

,B1 (선택 기준),B2 (점검),B3 (점검),전체
dq_var,-0.83,-0.71,-0.76,-0.88
dq_min,-0.79,-0.66,-0.74,-0.87
dq_mean,-0.76,-0.70,-0.72,-0.85
avgC,-0.43,0.32,-0.16,-0.06
Tavg_100,-0.12,0.17,0.02,0.30
QD_2,0.16,0.08,0.20,-0.55
IR_2,0.18,-0.37,0.19,-0.55
chargetime_5,0.43,-0.80,0.08,-0.26
fade_10_100,0.50,-0.27,0.16,-0.26


## 다중공선성 (B1)

In [3]:
def vif(X):
    X = (X - X.mean()) / X.std(); out = {}
    for c in X:
        A = np.c_[np.ones(len(X)), X.drop(columns=c).values]
        beta, *_ = np.linalg.lstsq(A, X[c].values, rcond=None)
        r2 = 1 - ((X[c].values - A @ beta) ** 2).sum() / ((X[c] - X[c].mean()) ** 2).sum()
        out[c] = 1 / (1 - r2)
    return pd.Series(out).round(1)
vif(df[df.batch=='b1'][['dq_var','dq_min','dq_mean','fade_10_100']])

dq_var         208.5
dq_min         415.3
dq_mean         74.3
fade_10_100      2.0
dtype: float64

## 결정 (DAY1 전략서와 동일)
| 세트 | 피처 | 근거 |
|---|---|---|
| A (기본) | dq_var | B1 최상위 |
| B (비교) | dq_var + fade_10_100 | fade B1 ρ 0.50 |
| C (대체) | dq_min | dq_var와 중복 |
| D (대조군) | A + B + QD_2, IR_2, Tavg_100, chargetime_5 | EDA에서 제외한 피처를 넣으면 실제로 나빠지는지 확인 |